[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/38_prefill_decode_separation.ipynb)

# 🔴 困难: Prefill-Decode 分离推理

实现 **Prefill-Decode分离** 的推理过程——将LLM推理分为预填充 (Prefill) 和解码 (Decode) 两个阶段，这是大模型高性能部署的核心技术。

**Prefill阶段**：一次性处理输入prompt的所有token，生成第一个输出token和KV Cache。

**Decode阶段**：自回归生成后续 token，每步只处理一个新 token，复用 KV Cache。

### 关键概念
- **KV Cache**: 存储每层的Key和Value张量，形状为 `(batch_size, num_kv_heads, seq_len, head_dim)`
- **因果掩码**: 确保token只能关注之前的token，Prefill使用下三角矩阵，Decode使用单元素掩码
- **位置编码**: 每个token需要正确的位置ID，Prefill阶段为 `[0, 1, ..., seq_len-1]`

### 函数签名
```python
def prefill_decode_inference(model, tokenizer, prompt: str, max_new_tokens: int = 128):
    # model: Qwen3ForCausalLM实例
    # tokenizer: AutoTokenizer实例
    # prompt: 输入文本
    # max_new_tokens: 最大生成token数
    # 返回: 生成的完整文本
```

### 要求
1. 正确实现Prefill和Decode两个阶段
2. 正确管理和更新KV Cache
3. 使用因果掩码保证自回归特性
4. 处理EOS token提前终止

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge transformers')
except ImportError:
    pass

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, Qwen3ForCausalLM
from transformers.models.qwen3 import Qwen3ForCausalLM  # 注意：这需要正确的导入

In [ ]:
# 辅助函数：创建因果掩码
def create_causal_mask(seq_len, device):
    """创建下三角因果掩码"""
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask

def create_single_token_mask(seq_len, pos, device):
    """为decode阶段创建单token掩码"""
    mask = torch.zeros(1, seq_len, device=device)
    mask[0, pos] = 1.0
    return mask

In [ ]:
# ✏️ 在此实现你的代码

def prefill_decode_inference(model, tokenizer, prompt: str, max_new_tokens: int = 128):
    # 1. 应用chat template
    # 2. Tokenize输入
    # 3. Prefill阶段：处理所有输入token
    # 4. 获取第一个生成token
    # 5. Decode阶段：自回归生成
    # 6. 返回完整生成的文本
    
    model.eval()
    device = next(model.parameters()).device
    
    # 在这里实现你的代码

In [ ]:
def prefill_decode_inference(model, tokenizer, prompt: str, max_new_tokens: int = 128):
    """
    执行Prefill-Decode分离推理

    Args:
        model: 模型
        tokenizer: 分词器
        prompt: 输入提示
        max_new_tokens: 最大生成长度
        
    Returns:
        生成的文本
    """
    model.eval()

    # 1. 预处理输入
    inputs = tokenizer(prompt, return_tensors="pt").cuda()
    input_ids = inputs["input_ids"]
    seq_len = input_ids.shape[1]

    # 2. 初始化KV Cache（使用模型默认的空cache）
    past_key_values = None

    # 3. ---- Prefill阶段 (一次性处理所有输入token) ----
    with torch.no_grad():
        # 使用位置ID: [0, 1, ..., seq_len-1]
        position_ids = torch.arange(seq_len).unsqueeze(0)
        
        # 第一次前向传播，生成第一个输出 token 和 KV Cache
        outputs = model(
            input_ids=input_ids,
            position_ids=position_ids,
            past_key_values=past_key_values,  # 初始为None
            use_cache=True,                    # 启用KV Cache返回
            return_dict=True
        )
    
    # 获取初始KV Cache和第一个输出logits
    past_key_values = outputs.past_key_values
    next_token_logits = outputs.logits[:, -1, :]  # 取最后一个 token 的 logits
    
    # 采样第一个生成的token
    next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)
    generated_ids = [next_token_id.item()]
    
    # 4. ---- Decode阶段 (自回归生成后续token) ----
    for step in range(max_new_tokens - 1):  # 第一个token已生成
        with torch.no_grad():
            # 当前要处理的新token位置ID为: 已处理的总长度
            current_pos = seq_len + step
            position_ids = torch.tensor([[current_pos]], dtype=torch.long)
            
            # 只输入最新的token，复用KV Cache
            outputs = model(
                input_ids=next_token_id,          # 形状: (1, 1)
                position_ids=position_ids,
                past_key_values=past_key_values,  # 传入缓存的KV
                use_cache=True,
                return_dict=True
            )
        
        # 更新KV Cache
        past_key_values = outputs.past_key_values
        
        # 获取下一个token的logits并采样
        next_token_logits = outputs.logits[:, -1, :]
        next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)
        
        # 检查是否遇到EOS token
        if next_token_id.item() == tokenizer.eos_token_id:
            break
            
        generated_ids.append(next_token_id.item())
    
    # 5. 解码生成的全部token (包含原始prompt和生成部分)
    full_ids = torch.cat([input_ids.squeeze(0), torch.tensor(generated_ids)])
    full_text = tokenizer.decode(full_ids, skip_special_tokens=True)
    
    return full_text

In [ ]:
# 🧪 调试 - 加载模型并测试

model_path = "/hy-tmp/hz/models-hub/Qwen3-0.6B"  # 或者使用HuggingFace路径
try:
    model = Qwen3ForCausalLM.from_pretrained(model_path).eval().cuda()
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    prompt = "Who is Donald Trump?"
    generated_text = prefill_decode_inference(model, tokenizer, prompt, max_new_tokens=50)
    print("Generated text:", generated_text)
except Exception as e:
    print(f"Error loading model: {e}")
    print("Make sure the model path is correct or use a different model")

In [ ]:
# ✅ 提交 - 验证实现
from torch_judge import check
check('prefill_decode_inference')